# Loading Datasets

## CircuitBench Tutorial 2

This notebook demonstrates publication-quality dataset loading, validation, preprocessing, metadata inspection, and reproducible data preparation for CircuitBench.

### Objectives

- Load benchmark datasets
- Inspect metadata
- Validate schema
- Handle missing values
- Detect duplicate samples
- Explore feature distributions
- Split datasets reproducibly
- Export processed datasets


## Scientific Background

Reliable benchmarking begins with reliable datasets.

This notebook demonstrates standardized loading procedures designed to maximize reproducibility while minimizing data leakage.


In [ ]:
from pathlib import Path
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
import circuitbench
from circuitbench.benchmark import BenchmarkRunner

runner = BenchmarkRunner()

## Available Benchmark Datasets

In [ ]:
datasets = runner.available_benchmarks()
datasets

## Dataset Categories

- Analog Circuits
- Digital Circuits
- RF Circuits
- Power Electronics
- Mixed-Signal Systems


In [ ]:
benchmark = runner.load(
    benchmark_name='CMOS_Inverter'
)

benchmark

## Dataset Metadata

In [ ]:
benchmark.summary()

## Feature Names

In [ ]:
benchmark.feature_names

## Target Variables

In [ ]:
benchmark.target_names

## Dataset Dimensions

In [ ]:
print('Samples :', benchmark.X.shape[0])
print('Features:', benchmark.X.shape[1])
print('Targets :', benchmark.y.shape)

## Preview Dataset

In [ ]:
benchmark.dataframe.head()

## Dataset Information

In [ ]:
benchmark.dataframe.info()

## Summary Statistics

In [ ]:
benchmark.dataframe.describe(include='all')

## Missing Value Analysis

In [ ]:
missing = benchmark.dataframe.isnull().sum()
missing[missing > 0]

## Duplicate Sample Detection

In [ ]:
duplicates = benchmark.dataframe.duplicated().sum()
print('Duplicate rows:', duplicates)

## Data Types

In [ ]:
benchmark.dataframe.dtypes

## Numerical Feature Distribution

In [ ]:
benchmark.dataframe.hist(figsize=(16,12))
plt.tight_layout()

## Correlation Matrix

In [ ]:
corr = benchmark.dataframe.corr(numeric_only=True)
corr

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(12,10))
plt.imshow(corr, interpolation='nearest', aspect='auto')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar(label='Correlation')
plt.title('Feature Correlation Matrix')
plt.tight_layout()

## Feature and Target Separation

In [ ]:
X = benchmark.X
y = benchmark.y

print('Feature matrix:', X.shape)
print('Target matrix:', y.shape)

## Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
)


## Dataset Split Summary

In [ ]:
split_summary = pd.DataFrame({
    'Subset':['Training','Validation','Testing'],
    'Samples':[len(X_train),len(X_valid),len(X_test)]
})

split_summary

## Saving Processed Dataset

In [ ]:
output_dir = Path('processed_data')
output_dir.mkdir(exist_ok=True)

pd.DataFrame(X_train).to_csv(output_dir/'X_train.csv', index=False)
pd.DataFrame(X_valid).to_csv(output_dir/'X_valid.csv', index=False)
pd.DataFrame(X_test).to_csv(output_dir/'X_test.csv', index=False)

pd.DataFrame(y_train).to_csv(output_dir/'y_train.csv', index=False)
pd.DataFrame(y_valid).to_csv(output_dir/'y_valid.csv', index=False)
pd.DataFrame(y_test).to_csv(output_dir/'y_test.csv', index=False)


## Export Dataset Metadata

In [ ]:
metadata = {
    'benchmark':'CMOS_Inverter',
    'random_seed':SEED,
    'training_samples':len(X_train),
    'validation_samples':len(X_valid),
    'testing_samples':len(X_test)
}

import json

with open(output_dir/'metadata.json','w') as f:
    json.dump(metadata,f,indent=4)


## Reproducibility Checklist

- Fixed random seed
- Dataset integrity verified
- Missing values inspected
- Duplicate samples checked
- Standardized train/validation/test split
- Metadata exported
- Processed datasets saved


## Best Practices

1. Never preprocess using the entire dataset before splitting.
2. Always report the random seed.
3. Preserve the original dataset.
4. Record benchmark metadata.
5. Export processed datasets for reproducibility.


## Next Notebook

Continue with **3_Running_SPICE_Simulations.ipynb** to learn how to execute SPICE simulations and generate benchmark datasets from circuit netlists.

# Summary

In this notebook we:

- Loaded a CircuitBench benchmark dataset.
- Explored benchmark metadata.
- Examined dataset dimensions and feature names.
- Checked data quality through missing-value and duplicate analysis.
- Visualized feature distributions and correlations.
- Created reproducible train/validation/test splits.
- Exported processed datasets and metadata for future experiments.

These steps establish a reproducible foundation for benchmarking surrogate machine-learning models in electronic circuit design.